# 06 — Streamlit Prediction Pipeline Demo

**CRISP-DM Phase 6 (Deployment) — companion notebook.**

This notebook demonstrates *exactly* what the Streamlit app does when a user
uploads a Sensor Logger recording, but in a notebook so the pipeline can be
inspected step by step.

It is **self-contained**: it generates a synthetic `WristMotion.csv` and runs
the shared `predict_from_sensor_logger()` function, so it works after a fresh
clone **without** the RecoFit dataset (the trained model is committed to git).

Pipeline: load CSV → normalize columns → sliding window → feature extraction
→ model prediction → confidence. The app calls the same function.

## 1. Load the committed model and feature names

These are the same artifacts the app loads via `st.cache_resource`.

In [ ]:
import joblib

from ml4b.utils.config import DATA_PROCESSED, MODELS_DIR

model = joblib.load(MODELS_DIR / "best_model.joblib")
feature_names = (DATA_PROCESSED / "feature_names.txt").read_text().strip().split("\n")
print(f"Loaded model: {type(model).__name__}")
print(f"Feature count: {len(feature_names)}")

## 2. Create a synthetic Sensor Logger `WristMotion.csv`

Format A is the Sensor Logger default: `time, seconds_elapsed, x, y, z,
roll, pitch, yaw`. In a real run you would upload your own recording instead.

In [ ]:
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 1000  # 1000 samples @ 50 Hz = 20 seconds
wrist = pd.DataFrame(
    {
        "time": np.arange(n),
        "seconds_elapsed": np.arange(n) / 50.0,
        "x": rng.standard_normal(n),
        "y": rng.standard_normal(n),
        "z": rng.standard_normal(n),
        "roll": rng.standard_normal(n),
        "pitch": rng.standard_normal(n),
        "yaw": rng.standard_normal(n),
    }
)
csv_path = Path(tempfile.gettempdir()) / "WristMotion.csv"
wrist.to_csv(csv_path, index=False)
wrist.head()

## 3. Run the shared prediction pipeline

`predict_from_sensor_logger()` is the **same** function the app calls. It also
accepts a Sensor Logger ZIP — just pass the `.zip` path instead.

In [ ]:
from ml4b.data.apple_watch_loader import predict_from_sensor_logger

results = predict_from_sensor_logger(csv_path, model, feature_names)
print(f"Windows predicted: {len(results)}")
results.head(10)

## 4. Summarize the predictions

This is the information the app turns into the timeline chart, the
distribution pie chart, and the results table.

In [ ]:
print("Predicted exercise distribution (per 2-second window):")
print(results["predicted_class"].value_counts().to_string())
print(f"\nMean confidence: {results['confidence'].mean():.3f}")
print(f"Total time covered: {results['time_start_seconds'].max() + 2:.0f} s")

## 5. (Optional) Evaluate on the held-out test set

If you have run the data-preparation pipeline (so `data/processed/` contains
the feature CSVs), this cell reproduces the headline test metric shown on the
app's Model Performance page. It is skipped automatically if the file is absent.

In [ ]:
from ml4b.models.evaluate import evaluate_model

test_csv = DATA_PROCESSED / "test_features.csv"
if test_csv.exists():
    test_df = pd.read_csv(test_csv)
    classes = sorted(test_df["exercise_name"].unique())
    res = evaluate_model(
        model,
        test_df[feature_names].values,
        test_df["exercise_name"].values,
        "Random Forest",
        classes,
    )
    print(f"Test Accuracy : {res['accuracy']:.4f}")
    print(f"Test Macro F1 : {res['macro_f1']:.4f}")
else:
    print("test_features.csv not found — run scripts/train_model.py to generate it.")